# Identity verification: a small experiment

This walkthrough trains a separate verifier for each of six people. Each model
learns only that person's enrolment beats. The probes contain equal numbers of
genuine beats and beats from other people.

The notebook calls the shared pipeline for training, scoring and thresholds.
It uses fewer people and epochs than the paper, so its results are a demonstration.
Run notebook 1 first to explore the signal processing.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = next(p.resolve() for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pipeline.py").exists())
sys.path.insert(0, str(REPO_ROOT))
import pipeline

cfg = pipeline.PipelineConfig()
cfg.db_dir = str(REPO_ROOT / "ecgiddb")
cfg.num_subjects = 6        # the paper uses 30
cfg.max_epochs = 60         # the paper uses 160, with early stopping
pipeline.set_all_seeds(cfg.seed)

N_PEOPLE = cfg.num_subjects
print(f"people in this demo: {N_PEOPLE}   (paper: 30)")
# Download ECG-ID only if it is not already available.
from ecg_features import load_ecg_id_from_physionet
if not load_ecg_id_from_physionet(cfg):
    raise RuntimeError("Download ECG-ID from PhysioNet before continuing")


## 1. Build both feature sets


In [ ]:
people = {p: pipeline.load_person(p, cfg) for p in range(1, N_PEOPLE + 1)}

for p, d in people.items():
    print(f"Person {p:02d}: {len(d['beats']):3d} beats")

print(f"\nwavelet summary: {people[1]['wtmm'].shape[1]} values per beat")
print(f"raw waveform   : {people[1]['raw'].shape[1]} values per beat")

## 2. Enrol one person and evaluate all six models

The evaluator fits scaling on enrolment beats, prepares balanced probes, and
sets each acceptance threshold from enrolment scores. The saved details show
the beat indices and scores for this demonstration.


In [ ]:
ENROLLED = 1
feats_raw = {p - 1: d["raw"] for p, d in people.items()}
feats_wtmm = {p - 1: d["wtmm"] for p, d in people.items()}
details = {}
one_results, reconstruction, rocs = pipeline.evaluate_subject(
    ENROLLED - 1, "wtmm", feats_raw, feats_wtmm, cfg, audit=details)

print("enrolment beats:", len(details["enrollment_beats"]))
print("genuine probes:", len(details["genuine_beats"]))
print("impostor probes:", len(details["impostor_beats"]))
print("model input size:", details["effective_input_dim"])


## 3. Inspect the thresholds and results

AUROC measures ranking across thresholds. F1 uses the enrolment threshold.
EER is the approximate point where false acceptance and rejection are closest.
Higher AUROC and F1 are better; lower EER is better.


In [ ]:
# All six models use the pipeline's enrolment-only preprocessing and thresholds.
# Inspect the actual settings and scores instead of reimplementing training.
for name, model in details["models"].items():
    if "error" in model:
        print(name, model["error"])
    else:
        print(name, "threshold:", round(model["threshold"], 5),
              "direction:", model["score_direction"])


In [ ]:
pd.DataFrame(one_results).T.rename(
    columns={"auroc": "AUROC", "f1": "F1", "eer": "EER"}).round(3)


## 4. Plot reconstruction errors and ROC curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
kan = reconstruction["KAN Autoencoder"]
for group, colour in [("genuine", "#2ca02c"), ("impostor", "#d62728")]:
    axes[0].hist(kan[group], bins=12, alpha=0.65, color=colour, label=group)
axes[0].set_xlabel("Reconstruction error")
axes[0].set_ylabel("Probe beats")
axes[0].set_title(f"KAN autoencoder, Person {ENROLLED:02d}")
axes[0].legend()
for name, (fpr, tpr, auc) in rocs.items():
    axes[1].plot(fpr, tpr, label=f"{name} ({auc:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5)
axes[1].set_xlabel("False accept rate")
axes[1].set_ylabel("True accept rate")
axes[1].set_title("ROC curves")
axes[1].legend(fontsize=7)
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()


## 5. Repeat for all six people and both input branches

This is a separate demonstration run. The neural random sequence differs from
the archived 30-person experiment.


In [ ]:
rows = []
for feature_set in ("raw", "wtmm"):
    for idx in sorted(feats_wtmm):
        results, _, _ = pipeline.evaluate_subject(
            idx, feature_set, feats_raw, feats_wtmm, cfg)
        for model, metrics in results.items():
            rows.append({"feature_set": feature_set, "person": idx + 1,
                         "model": model, **metrics})
summary = (pd.DataFrame(rows)
           .groupby(["feature_set", "model"])[["auroc", "f1", "eer"]]
           .mean().round(3))
summary


The published metrics are in `results/`. The manuscript explains the limits
of this small, within-recording evaluation. A separate recording is needed to
test changes in acquisition conditions.
